# 00 · Verificación del entorno — los 6 modelos en un solo cuaderno

Comprueba que el **mismo conjunto de versiones** funciona con los 6 LLMs, en la evaluación base y en el ajuste fino QLoRA.

**Requisitos del pod:** RTX 5090, **Volume Disk de 200 GB** (los 6 modelos ocupan ~110 GB), variable de entorno `HF_TOKEN` y `requirements.txt` en `/workspace`. Kernel: **Python 3**.

**Tiempo estimado:** 30–60 min, casi todo en la descarga de los modelos.

## 0. Instalación del entorno
Para hacer la verificacion hacemos lo siguiente:
1. Corre la celda (1–2 min).
2. **Reinicia el kernel** (*Kernel → Restart*).
3. Sigue desde la sección 1.

Se repite cada vez que el pod se enciende de nuevo.

In [ ]:
%pip install -q -r /workspace/requirements.txt

## 1. Versiones, CUDA, GPU y disco

In [ ]:
import os, platform, shutil, subprocess, importlib.metadata as md_
import torch

ESPERADAS = {
    "torch": "2.8.0", "torchvision": "0.23.0", "transformers": "5.15.0", "tokenizers": "0.22.2",
    "huggingface_hub": "1.27.0", "safetensors": "0.8.0", "Jinja2": "3.1.6", "pillow": "12.3.0",
    "sentencepiece": "0.2.2", "protobuf": "7.35.1", "peft": "0.20.0", "bitsandbytes": "0.50.1",
    "trl": "1.10.0", "accelerate": "1.14.0", "datasets": "5.0.1", "fsspec": "2026.6.0",
    "pyarrow": "25.0.1", "pandas": "3.0.5", "numpy": "2.5.2", "scikit-learn": "1.9.0",
    "matplotlib": "3.11.1", "seaborn": "0.13.2", "tqdm": "4.70.0", "ipywidgets": "8.1.8",
}
errores = []
for paquete, esperada in ESPERADAS.items():
    try:
        instalada = md_.version(paquete)
    except md_.PackageNotFoundError:
        instalada = "NO INSTALADA"
    ok = instalada.split("+")[0] == esperada
    print(f"{'OK ' if ok else 'XX '} {paquete:16s} {instalada:20s} (esperada {esperada})")
    if not ok: errores.append(paquete)

driver = subprocess.run(["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
                        capture_output=True, text=True).stdout.strip()
print(f"\nPython {platform.python_version()} | torch {torch.__version__} | CUDA {torch.version.cuda} "
      f"| cuDNN {torch.backends.cudnn.version()} | driver {driver}")
print("GPU:", torch.cuda.get_device_name(0), "| bf16:", torch.cuda.is_bf16_supported())
total, _, libre = shutil.disk_usage("/workspace")
print(f"Disco /workspace: {libre/1e9:.0f} GB libres de {total/1e9:.0f} GB")

assert not errores, f"Versiones distintas a las fijadas: {errores}"
assert torch.version.cuda == "12.8", f"CUDA distinta a la fijada: {torch.version.cuda}"
assert os.environ.get("HF_TOKEN"), "Configura HF_TOKEN como variable de entorno del pod"
if libre < 130e9:
    print("AVISO: menos de 130 GB libres; puede no alcanzar para los 6 modelos")

## 2. Script de verificación de un modelo

Esta celda escribe `/workspace/verificar_modelo.py`. Recibe el nombre de un modelo y hace:
1. Plantilla de chat y máscara de pérdida (prompts 1 y 4).
2. Evaluación base: bf16, genera 3 respuestas.
3. Ajuste fino: 4 bits + LoRA, 2 pasos de `SFTTrainer`.
4. Guarda el resultado en `/workspace/verificacion/<modelo>.json`.

In [ ]:
%%writefile /workspace/verificar_modelo.py
import os, sys, json, time, gc, platform, subprocess, shutil, traceback
os.environ.setdefault("HF_HOME", "/workspace/hf_cache")
import torch

MODELOS = {
    "qwen3_5_9b":     dict(id="Qwen/Qwen3.5-9B",                    rev="c202236235762e1c871ad0ccb60c8ee5ba337b9a", tipo="multimodal", thinking=True),
    "rigochat_7b_v2": dict(id="IIC/RigoChat-7b-v2",                 rev="666a48562f47b0cc09aca23f1ae8df3d0b3ac25c", tipo="causal_lm",  thinking=False),
    "salamandra_7b":  dict(id="BSC-LT/salamandra-7b-instruct-2606", rev="91055ea2c907ddcf28209d2cb4e078a093ea44d8", tipo="causal_lm",  thinking=False),
    "eurollm_9b":     dict(id="utter-project/EuroLLM-9B-Instruct",  rev="f7ae2bc3bcbb538c0b93fa6cfbf388d9898b1ace", tipo="causal_lm",  thinking=False),
    "gemma4_12b_it":  dict(id="google/gemma-4-12B-it",              rev="707f0a3b8a3c7ad586ed01e27eafbad8a27dd0f7", tipo="multimodal", thinking=True),
    "llama3_1_8b":    dict(id="meta-llama/Llama-3.1-8B-Instruct",   rev="0e9e39f249a16976918f6564b8830bc894c89659", tipo="causal_lm",  thinking=False),
}
PROMPTS = {
    1: {"system": None,
        "user": "Clasifica el sentimiento del siguiente comentario. Responde solo con una de estas palabras: negativo, neutral, positivo. Comentario: {text}"},
    4: {"system": "Eres un clasificador de sentimientos de comentarios en espanol peruano. Responde unicamente con una de las siguientes clases: negativo, neutral, positivo.",
        "user": "Clasifica el siguiente comentario. Comentario: {text}"},
}
TEXTO, ETIQUETA = "Los de la U también son del Orto Equipo 😘", "negativo"
TEXTOS = [TEXTO, "Qué buena chamba la de hoy, todo salió chévere", "Mañana hay reunión a las 10"]

nombre = sys.argv[1]
cfg = MODELOS[nombre]
hf_token = os.environ["HF_TOKEN"]
driver = subprocess.run(["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
                        capture_output=True, text=True).stdout.strip()
import transformers, trl, peft, bitsandbytes
res = dict(modelo=cfg["id"], nombre=nombre, revision=cfg["rev"], python=platform.python_version(),
           torch=torch.__version__, cuda=torch.version.cuda, cudnn=torch.backends.cudnn.version(),
           gpu=torch.cuda.get_device_name(0), driver=driver, transformers=transformers.__version__,
           trl=trl.__version__, peft=peft.__version__, bitsandbytes=bitsandbytes.__version__)

def guardar():
    os.makedirs("/workspace/verificacion", exist_ok=True)
    with open(f"/workspace/verificacion/{nombre}.json", "w") as f:
        json.dump(res, f, ensure_ascii=False, indent=1)

try:
    from transformers import (AutoTokenizer, AutoProcessor, AutoModelForCausalLM,
                              AutoModelForMultimodalLM, BitsAndBytesConfig)
    if cfg["tipo"] == "multimodal":
        plantilla = AutoProcessor.from_pretrained(cfg["id"], revision=cfg["rev"], token=hf_token)
        tokenizer = plantilla.tokenizer
    else:
        plantilla = tokenizer = AutoTokenizer.from_pretrained(cfg["id"], revision=cfg["rev"], token=hf_token)
    Clase = AutoModelForMultimodalLM if cfg["tipo"] == "multimodal" else AutoModelForCausalLM
    KW = {"enable_thinking": False} if cfg["thinking"] else {}

    def mensajes(pid, texto):
        c = PROMPTS[pid]
        m = [{"role": "system", "content": c["system"]}] if c["system"] else []
        return m + [{"role": "user", "content": c["user"].format(text=texto)}]

    def ejemplo(pid, texto, etiqueta):
        m_p = mensajes(pid, texto)
        p_txt = plantilla.apply_chat_template(m_p, add_generation_prompt=True, tokenize=False, **KW)
        f_txt = plantilla.apply_chat_template(m_p + [{"role": "assistant", "content": etiqueta}], tokenize=False)
        c_txt = f_txt[f_txt.rfind(etiqueta):]
        p_ids = tokenizer(p_txt, add_special_tokens=False)["input_ids"]
        c_ids = tokenizer(c_txt, add_special_tokens=False)["input_ids"]
        return {"input_ids": p_ids + c_ids, "labels": [-100] * len(p_ids) + c_ids}

    # 1. Plantilla y mascara
    for pid in (1, 4):
        ej = ejemplo(pid, TEXTO, ETIQUETA)
        en = [t for t, l in zip(ej["input_ids"], ej["labels"]) if l != -100]
        res[f"p{pid}_secuencia"] = tokenizer.decode(ej["input_ids"])
        res[f"p{pid}_en_perdida"] = tokenizer.decode(en)
        res[f"p{pid}_tokens_perdida"] = f"{len(en)}/{len(ej['input_ids'])}"
    res["plantilla_ok"] = True
    print("Plantilla OK:", res["p1_en_perdida"].__repr__())

    # 2. Evaluacion base (bf16)
    torch.cuda.reset_peak_memory_stats(); t0 = time.time()
    model = Clase.from_pretrained(cfg["id"], revision=cfg["rev"], token=hf_token,
                                  dtype=torch.bfloat16, device_map="auto")
    pad = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    salidas = []
    for t in TEXTOS:
        p = plantilla.apply_chat_template(mensajes(1, t), add_generation_prompt=True, tokenize=False, **KW)
        ids = tokenizer(p, add_special_tokens=False, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**ids, max_new_tokens=10, do_sample=False, pad_token_id=pad)
        salidas.append(tokenizer.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True))
    res.update(base_ok=True, base_salidas=salidas, base_mem_gb=round(torch.cuda.max_memory_allocated() / 1e9, 2),
               base_seg=round(time.time() - t0))
    print("Base OK:", salidas)
    del model; gc.collect(); torch.cuda.empty_cache()

    # 3. Ajuste fino QLoRA (2 pasos)
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from trl import SFTConfig, SFTTrainer
    from datasets import Dataset
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    torch.cuda.reset_peak_memory_stats(); t0 = time.time()
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
    model = Clase.from_pretrained(cfg["id"], revision=cfg["rev"], token=hf_token,
                                  quantization_config=bnb, device_map="auto")
    model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, LoraConfig(r=16, lora_alpha=16, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
    entrenables = sum(p.numel() for p in model.parameters() if p.requires_grad)
    args = SFTConfig(output_dir=f"/workspace/smoke_{nombre}", max_steps=2, per_device_train_batch_size=8,
                     learning_rate=2e-4, bf16=True, logging_steps=1, report_to="none", save_strategy="no",
                     max_length=512, packing=False, gradient_checkpointing=False)
    trainer = SFTTrainer(model=model, args=args, train_dataset=Dataset.from_list([ejemplo(1, TEXTO, ETIQUETA)] * 16),
                         processing_class=tokenizer)
    out = trainer.train()
    res.update(qlora_ok=True, qlora_loss=round(out.training_loss, 4), qlora_parametros_entrenables=entrenables,
               qlora_mem_gb=round(torch.cuda.max_memory_allocated() / 1e9, 2), qlora_seg=round(time.time() - t0))
    print("QLoRA OK: loss", res["qlora_loss"])
    shutil.rmtree(f"/workspace/smoke_{nombre}", ignore_errors=True)
except Exception as e:
    res["error"] = f"{type(e).__name__}: {e}"
    traceback.print_exc()
finally:
    guardar()

## 3. Ejecutar los 6 modelos

Cada modelo corre en su propio proceso; la salida se muestra en vivo. Para probar solo algunos, edita la lista `A_PROBAR`.

In [ ]:
import json, shutil, subprocess, sys, os

A_PROBAR = ["qwen3_5_9b", "rigochat_7b_v2", "salamandra_7b", "eurollm_9b", "gemma4_12b_it", "llama3_1_8b"]

for nombre in A_PROBAR:
    libre = shutil.disk_usage("/workspace").free / 1e9
    print("=" * 100, f"\n{nombre}  |  disco libre: {libre:.0f} GB", flush=True)
    if libre < 30:
        print("  SALTADO: menos de 30 GB libres"); continue
    proc = subprocess.run([sys.executable, "/workspace/verificar_modelo.py", nombre])
    print(f"  proceso terminado (codigo {proc.returncode})")

## 4. Resumen

In [ ]:
import pandas as pd, glob
filas = [json.load(open(f)) for f in sorted(glob.glob("/workspace/verificacion/*.json"))]
df = pd.DataFrame(filas)
cols = ["nombre", "plantilla_ok", "p1_en_perdida", "p4_en_perdida", "base_ok", "base_salidas", "base_mem_gb",
        "qlora_ok", "qlora_loss", "qlora_parametros_entrenables", "qlora_mem_gb", "error"]
display(df[[c for c in cols if c in df.columns]])

# Mismo entorno en todos los modelos?
cols_entorno = ["python", "torch", "cuda", "cudnn", "gpu", "driver", "transformers", "trl", "peft", "bitsandbytes"]
print("Entorno identico en todos los modelos:", (df[cols_entorno].nunique() == 1).all())
display(df[["nombre"] + cols_entorno])
df.to_csv("/workspace/verificacion/resumen_verificacion.csv", index=False)

## 5. Registro del entorno (para la tesis y el repositorio)

In [ ]:
!pip freeze > /workspace/verificacion/entorno_congelado.txt
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv > /workspace/verificacion/gpu_info.csv
print(open("/workspace/verificacion/gpu_info.csv").read())